# TCGA-BRCA Drug Name Normalization Review V1

This notebook is review-only. It reads the latest saved drug-name normalization outputs from disk,
regenerates review tables in `05-results`, and does not reread raw treatment tables, modify raw rows,
freeze treatment arms, perform modeling, or make treatment recommendations.


In [ ]:
from pathlib import Path
import json

import pandas as pd
from IPython.display import display


def detect_repo_root(start_path: Path) -> Path:
    for candidate in [start_path, *start_path.parents]:
        if (candidate / '.git').exists():
            return candidate
    raise FileNotFoundError('Unable to locate the repository root from the notebook path.')


def read_tsv(path: Path) -> pd.DataFrame:
    return pd.read_csv(path, sep='\t', dtype=str, keep_default_na=False)


repo_root = detect_repo_root(Path.cwd())
latest_pointer_path = (
    repo_root
    / '01-data'
    / 'audit'
    / 'tcga-brca'
    / 'treatment-prep'
    / 'tcga_brca_drug_name_normalization_v1_latest.json'
)
if not latest_pointer_path.exists():
    raise FileNotFoundError(
        f'Latest drug-name normalization v1 pointer not found: {latest_pointer_path}. '
        'Run script 23 first.'
    )

latest_pointer = json.loads(latest_pointer_path.read_text(encoding='utf-8'))
inventory_path = repo_root / latest_pointer['drug_name_inventory_v1_tsv']
normalization_map_path = repo_root / latest_pointer['drug_name_normalization_map_v1_tsv']
class_map_path = repo_root / latest_pointer['drug_name_class_map_v1_tsv']
patient_profile_path = repo_root / latest_pointer['patient_drug_class_profile_v1_tsv']
summary_path = repo_root / latest_pointer['drug_name_normalization_v1_summary_tsv']
run_log_path = repo_root / latest_pointer['run_log_json']

for required_path in [
    inventory_path,
    normalization_map_path,
    class_map_path,
    patient_profile_path,
    summary_path,
    run_log_path,
]:
    if not required_path.exists():
        raise FileNotFoundError(f'Required drug-name normalization artifact not found: {required_path}')

inventory_df = read_tsv(inventory_path)
normalization_map_df = read_tsv(normalization_map_path)
class_map_df = read_tsv(class_map_path)
patient_profile_df = read_tsv(patient_profile_path)
summary_df = read_tsv(summary_path)
run_log = json.loads(run_log_path.read_text(encoding='utf-8'))

if run_log.get('status') != 'completed':
    raise ValueError('run_log.json does not report status == completed.')
if not bool(run_log.get('validation', {}).get('passed', False)):
    raise ValueError('run_log.json does not report validation.passed == true.')
if inventory_df.empty:
    raise ValueError('drug_name_inventory_v1.tsv contains no rows.')

results_root = (
    repo_root
    / '09-trials'
    / '01-tcga-only-source-audited'
    / '05-results'
)
results_root.mkdir(parents=True, exist_ok=True)

print(f"Run ID      : {latest_pointer['drug_name_normalization_v1_run_id']}")
print(f"Profile ID  : {latest_pointer['patient_treatment_profile_v1_run_id']}")
print(f"Grouping ID : {latest_pointer['patient_treatment_grouping_v1_run_id']}")
print(f"Pointer     : {latest_pointer_path}")


In [ ]:
# --- Write review tables to 05-results/ ---

review_inventory_path = results_root / '126_drug_name_inventory_v1.tsv'
review_normalization_map_path = results_root / '127_drug_name_normalization_map_v1.tsv'
review_class_map_path = results_root / '128_drug_name_class_map_v1.tsv'
review_patient_profile_path = results_root / '129_patient_drug_class_profile_v1.tsv'
review_summary_path = results_root / '130_drug_name_normalization_v1_summary.tsv'

inventory_df.to_csv(review_inventory_path, sep='\t', index=False)
normalization_map_df.to_csv(review_normalization_map_path, sep='\t', index=False)
class_map_df.to_csv(review_class_map_path, sep='\t', index=False)
patient_profile_df.to_csv(review_patient_profile_path, sep='\t', index=False)
summary_df.to_csv(review_summary_path, sep='\t', index=False)

print(f'Saved: {review_inventory_path}')
print(f'Saved: {review_normalization_map_path}')
print(f'Saved: {review_class_map_path}')
print(f'Saved: {review_patient_profile_path}')
print(f'Saved: {review_summary_path}')


In [ ]:
print('=== Summary ===')
display(summary_df)

inventory_status_counts_df = (
    inventory_df
    .groupby('mapping_status', as_index=False)
    .agg(
        distinct_raw_drug_name_count=('raw_drug_name', 'size'),
        drug_row_count=('drug_row_count', lambda s: s.astype(int).sum()),
        distinct_patient_count=('distinct_patient_count', lambda s: s.astype(int).sum()),
    )
    .sort_values('mapping_status')
    .reset_index(drop=True)
)
print('\n=== Inventory status counts ===')
display(inventory_status_counts_df)

print('\n=== Class map preview ===')
display(class_map_df.head(30))


In [ ]:
top_unresolved_inventory_df = inventory_df[
    inventory_df['mapping_status'] != 'mapped_confidently'
].copy()
top_unresolved_inventory_df['drug_row_count'] = top_unresolved_inventory_df['drug_row_count'].astype(int)
top_unresolved_inventory_df = top_unresolved_inventory_df.sort_values(
    ['drug_row_count', 'mapping_status', 'raw_drug_name'], ascending=[False, True, True]
).reset_index(drop=True)
print('=== Top unresolved raw names ===')
display(top_unresolved_inventory_df.head(30))

review_patient_profile_df = patient_profile_df[
    (patient_profile_df['drug_name_normalization_requires_manual_review'] == 'yes')
    | (patient_profile_df['drug_class_mapping_requires_manual_review'] == 'yes')
].reset_index(drop=True)
print('\n=== Patient rows requiring review ===')
display(review_patient_profile_df.head(30))
